In [ ]:
# ============================================================
# Compare single-cell embeddings with:
#   1. Cell type conservation (ARI / NMI)
#   2. Batch effect correction
#
# Embeddings supported:
#   - scGPT
#   - STACK
#   - nicheformer
#
# Directory structure expected:
#
# master_dir/
#   NCBI783/
#       adata.h5ad
#       embeddings/
#           scGPT.csv
#           STACK.parquet
#           nicheformer.parquet
#
# ============================================================

# ============================================================
# Notebook Cell 1 — Imports
# ============================================================

import os
import glob
import warnings

import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad

from sklearn.cluster import KMeans
from sklearn.metrics import (
    adjusted_rand_score,
    normalized_mutual_info_score,
)

from sklearn.preprocessing import LabelEncoder

import scib
import scib.metrics as sm

from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA

from sklearn.cluster import (
    KMeans,
    MiniBatchKMeans,
)
warnings.filterwarnings("ignore")

print("scanpy:", sc.__version__)
print("scib:", scib.__version__)

In [ ]:
# ============================================================
# Notebook Cell — Add configurable subsampling
# ============================================================

# Number of cells to use PER SAMPLE
# Recommended:
#   25k  -> quick sanity check
#   50k  -> good balance
#   100k -> strong estimate
#
# With 11M total cells:
# DO NOT run metrics on all cells.
#
MAX_CELLS_PER_SAMPLE = 2500

# Optional:
# Keep cell type proportions approximately balanced
STRATIFIED_SUBSAMPLING = True

# Random seed
SUBSAMPLE_RANDOM_STATE = 42

In [ ]:
# ============================================================
# Notebook Cell 2 — Configuration
# ============================================================

MASTER_DIR = "../../../Broad_SpatialFoundation/hest_processed_data/"

EMBEDDING_FILES = {
    "scGPT": "scGPT.parquet",
    "STACK": "STACK.parquet",
    "nicheformer": "nicheformer.parquet",
}

# Column names in adata.obs
CELLTYPE_COLUMN_CANDIDATES = [
    "cell_type",
    "celltype",
    "celltypes",
    "CellType",
    "annotation",
    "labels",
]

BATCH_COLUMN = "batch"

# Clustering parameters
N_CLUSTERS = None  # if None -> number of true cell types
RANDOM_STATE = 0

# ------------------------------------------------------------
# Embedding optimization
# ------------------------------------------------------------

USE_PCA = False
PCA_COMPONENTS = 50

RANDOM_STATE = 0

In [ ]:
# ============================================================
# Notebook Cell — Subsampling helper
# ============================================================

from sklearn.model_selection import train_test_split


def subsample_adata(
    adata,
    label_key,
    max_cells=50000,
    stratified=True,
    random_state=42,
):
    """
    Subsample AnnData object.

    Parameters
    ----------
    adata : AnnData
    label_key : str
        Cell type column
    max_cells : int
    stratified : bool
        Preserve cell type proportions
    """

    n_cells = adata.n_obs

    if n_cells <= max_cells:
        return adata.copy()

    idx = np.arange(n_cells)

    if stratified:

        labels = adata.obs[label_key].astype(str)

        sampled_idx, _ = train_test_split(
            idx,
            train_size=max_cells,
            stratify=labels,
            random_state=random_state,
        )

    else:

        rng = np.random.default_rng(random_state)

        sampled_idx = rng.choice(
            idx,
            size=max_cells,
            replace=False,
        )

    sampled_idx = np.sort(sampled_idx)

    return adata[sampled_idx].copy()

In [ ]:
# ============================================================
# Notebook Cell — Helper Functions
# ============================================================

def find_celltype_column(obs):

    for col in CELLTYPE_COLUMN_CANDIDATES:
        if col in obs.columns:
            return col

    raise ValueError(
        f"Could not find cell type column.\n"
        f"Available columns:\n{list(obs.columns)}"
    )


def load_embedding(path):

    if path.endswith(".csv"):
        emb = pd.read_csv(path, index_col=0)

    elif path.endswith(".parquet"):
        emb = pd.read_parquet(path)

    else:
        raise ValueError(f"Unsupported format: {path}")

    return emb


def subsample_adata(
    adata,
    label_key="celltype",
    max_cells=50000,
    stratified=True,
    random_state=0,
):

    if adata.n_obs <= max_cells:
        return adata

    idx = np.arange(adata.n_obs)

    if stratified:

        labels = adata.obs[label_key].astype(str)

        # Count class frequencies
        counts = labels.value_counts()

        # Keep only classes with >=2 samples
        valid_classes = counts[counts >= 2].index

        valid_mask = labels.isin(valid_classes)

        if valid_mask.sum() < max_cells:
            print(
                f"Warning: after removing singleton classes, "
                f"only {valid_mask.sum()} cells remain."
            )

        idx_valid = idx[valid_mask]
        labels_valid = labels[valid_mask]

        sampled_idx, _ = train_test_split(
            idx_valid,
            train_size=min(max_cells, len(idx_valid)),
            stratify=labels_valid,
            random_state=random_state,
        )

    else:

        rng = np.random.default_rng(random_state)

        sampled_idx = rng.choice(
            idx,
            size=max_cells,
            replace=False,
        )

    return adata[sampled_idx].copy()


def preprocess_embedding(X):

    X = X.astype(np.float32)

    if USE_PCA and X.shape[1] > PCA_COMPONENTS:

        X = PCA(
            n_components=PCA_COMPONENTS,
            random_state=RANDOM_STATE,
        ).fit_transform(X)

    return X


def compute_ari_nmi(
    X,
    labels,
):

    n_clusters = len(np.unique(labels))

    km = MiniBatchKMeans(
        n_clusters=n_clusters,
        random_state=RANDOM_STATE,
        batch_size=4096,
        n_init="auto",
    )

    pred = km.fit_predict(X)

    ari = adjusted_rand_score(labels, pred)

    nmi = normalized_mutual_info_score(labels, pred)

    return ari, nmi


def compute_silhouette_metrics(
    adata,
    embed_key,
    batch_key,
    label_key,
):

    results = {}

    # --------------------------------------------------------
    # Batch mixing
    #
    # Lower absolute value = better batch mixing
    #
    # scIB rescales to:
    #   0 -> bad
    #   1 -> good
    # --------------------------------------------------------

    try:

        sb = sm.silhouette_batch(
            adata,
            batch_key=batch_key,
            label_key=label_key,
            embed=embed_key,
            verbose=False,
        )

        results["silhouette_batch"] = sb

    except Exception as e:

        print("silhouette_batch failed:", e)

        results["silhouette_batch"] = np.nan

    # --------------------------------------------------------
    # Biological conservation
    #
    # Higher = better
    # --------------------------------------------------------

    try:

        sl = sm.silhouette(
            adata,
            label_key=label_key,
            embed=embed_key,
        )

        results["silhouette_label"] = sl

    except Exception as e:

        print("silhouette_label failed:", e)

        results["silhouette_label"] = np.nan

    return results

In [ ]:
#sample_dirs = sorted([
#    os.path.join(MASTER_DIR, d)
#    for d in os.listdir(MASTER_DIR)
#    if os.path.isdir(os.path.join(MASTER_DIR, d))
#    and os.path.exists(
#        os.path.join(MASTER_DIR, d, "adata.h5ad")
#    )
#])

sample_dirs = ['../../../Broad_SpatialFoundation/hest_processed_data/NCBI783',
 '../../../Broad_SpatialFoundation/hest_processed_data/NCBI784',
 '../../../Broad_SpatialFoundation/hest_processed_data/NCBI785',
 '../../../Broad_SpatialFoundation/hest_processed_data/NCBI856',
 '../../../Broad_SpatialFoundation/hest_processed_data/NCBI857',
 '../../../Broad_SpatialFoundation/hest_processed_data/NCBI858',
 '../../../Broad_SpatialFoundation/hest_processed_data/NCBI859',
 '../../../Broad_SpatialFoundation/hest_processed_data/NCBI860',
 '../../../Broad_SpatialFoundation/hest_processed_data/NCBI861',
 '../../../Broad_SpatialFoundation/hest_processed_data/NCBI864',
 '../../../Broad_SpatialFoundation/hest_processed_data/NCBI865',
 '../../../Broad_SpatialFoundation/hest_processed_data/NCBI866',
 '../../../Broad_SpatialFoundation/hest_processed_data/NCBI867',
 '../../../Broad_SpatialFoundation/hest_processed_data/NCBI870',
 '../../../Broad_SpatialFoundation/hest_processed_data/NCBI873',
 '../../../Broad_SpatialFoundation/hest_processed_data/NCBI875',
 '../../../Broad_SpatialFoundation/hest_processed_data/NCBI876',
 '../../../Broad_SpatialFoundation/hest_processed_data/NCBI879',
 '../../../Broad_SpatialFoundation/hest_processed_data/NCBI880',
 '../../../Broad_SpatialFoundation/hest_processed_data/NCBI881',
 '../../../Broad_SpatialFoundation/hest_processed_data/NCBI882',
 '../../../Broad_SpatialFoundation/hest_processed_data/NCBI883',
 '../../../Broad_SpatialFoundation/hest_processed_data/NCBI884',
 '../../../Broad_SpatialFoundation/hest_processed_data/TENX105',
 '../../../Broad_SpatialFoundation/hest_processed_data/TENX106',
 '../../../Broad_SpatialFoundation/hest_processed_data/TENX111',
 '../../../Broad_SpatialFoundation/hest_processed_data/TENX114',
 '../../../Broad_SpatialFoundation/hest_processed_data/TENX115',
 '../../../Broad_SpatialFoundation/hest_processed_data/TENX116',
 '../../../Broad_SpatialFoundation/hest_processed_data/TENX117',
 '../../../Broad_SpatialFoundation/hest_processed_data/TENX118',
 '../../../Broad_SpatialFoundation/hest_processed_data/TENX119',
 '../../../Broad_SpatialFoundation/hest_processed_data/TENX120',]

print(f"Found {len(sample_dirs)} samples")

In [ ]:
# ============================================================
# Notebook Cell — Build Global AnnData WITH embeddings
# Robust to missing embeddings / partial overlap
# ============================================================

import scipy.sparse as sp

all_adatas = []

for sample_dir in sample_dirs:

    sample_name = os.path.basename(sample_dir)

    print("\n" + "=" * 70)
    print(f"Loading {sample_name}")
    print("=" * 70)

    # --------------------------------------------------------
    # Paths
    # --------------------------------------------------------

    adata_path = os.path.join(sample_dir, "adata.h5ad")
    ct_path = os.path.join(sample_dir, "celltypes.csv")
    emb_dir = os.path.join(sample_dir, "embeddings")

    if not os.path.exists(adata_path):
        print("Missing adata.h5ad")
        continue

    if not os.path.exists(ct_path):
        print("Missing celltypes.csv")
        continue

    # ========================================================
    # Load AnnData
    # ========================================================

    adata = sc.read_h5ad(adata_path)

    print(f"Loaded {adata.n_obs:,} cells")

    # ========================================================
    # Load celltypes
    # ========================================================

    ct_df = pd.read_csv(
        ct_path,
        index_col=0,
    )

    # Ensure string indices
    adata.obs_names = adata.obs_names.astype(str)
    ct_df.index = ct_df.index.astype(str)

    # Safe alignment
    ct_df = ct_df.reindex(adata.obs_names)

    adata.obs = pd.concat(
        [adata.obs, ct_df],
        axis=1,
    )

    # ========================================================
    # Detect celltype column
    # ========================================================

    celltype_col = find_celltype_column(
        adata.obs
    )

    print(f"Using cell type column: {celltype_col}")

    # ========================================================
    # Add batch column
    # ========================================================

    adata.obs["batch"] = sample_name

    # ========================================================
    # Standardize labels
    # ========================================================

    adata.obs["celltype"] = (
        adata.obs[celltype_col]
        .astype(str)
    )

    # ========================================================
    # Subsample ONCE
    # ========================================================

    original_n = adata.n_obs

    adata = subsample_adata(
        adata,
        label_key="celltype",
        max_cells=MAX_CELLS_PER_SAMPLE,
        stratified=STRATIFIED_SUBSAMPLING,
        random_state=SUBSAMPLE_RANDOM_STATE,
    )

    print(
        f"Subsampled "
        f"{original_n:,} -> {adata.n_obs:,} cells"
    )

    # ========================================================
    # LOAD EMBEDDINGS
    # ========================================================

    for emb_name, emb_file in EMBEDDING_FILES.items():

        print("\n" + "-" * 50)
        print(f"Loading {emb_name}")
        print("-" * 50)

        emb_path = os.path.join(
            emb_dir,
            emb_file,
        )

        # ----------------------------------------------------
        # Missing embedding file
        # ----------------------------------------------------

        if not os.path.exists(emb_path):

            print(f"Missing embedding file")

            adata.obsm[f"X_{emb_name}"] = np.full(
                (adata.n_obs, 1),
                np.nan,
                dtype=np.float32,
            )

            continue

        # ----------------------------------------------------
        # Load embedding
        # ----------------------------------------------------

        emb = load_embedding(emb_path)

        # Ensure string indices
        emb.index = emb.index.astype(str)

        # ====================================================
        # Reindex to match adata order
        #
        # Missing cells -> NaN
        # ====================================================

        emb = emb.reindex(
            adata.obs_names.astype(str)
        )

        # ----------------------------------------------------
        # Report missing embeddings
        # ----------------------------------------------------

        missing_mask = emb.isna().all(axis=1)

        n_missing = missing_mask.sum()

        if n_missing > 0:

            print(
                f"{n_missing:,} cells "
                f"missing embeddings"
            )

        # ----------------------------------------------------
        # Convert to float32
        # ----------------------------------------------------

        X = emb.values.astype(np.float32)

        # ====================================================
        # PCA compression
        #
        # IMPORTANT:
        # only run PCA on valid rows
        # ====================================================

        valid_mask = ~np.isnan(X).all(axis=1)

        if valid_mask.sum() == 0:

            print("No valid embeddings found")

            adata.obsm[f"X_{emb_name}"] = X

            continue

        if (
            USE_PCA
            and X.shape[1] > PCA_COMPONENTS
        ):

            X_valid = X[valid_mask]

            X_valid = PCA(
                n_components=PCA_COMPONENTS,
                random_state=RANDOM_STATE,
            ).fit_transform(X_valid)

            # Reinsert into NaN matrix
            X_pca = np.full(
                (
                    X.shape[0],
                    PCA_COMPONENTS,
                ),
                np.nan,
                dtype=np.float32,
            )

            X_pca[valid_mask] = X_valid

            X = X_pca

        # ====================================================
        # Store embedding
        # ====================================================

        adata.obsm[f"X_{emb_name}"] = X

        print(
            f"Stored {emb_name}: "
            f"{X.shape}"
        )

    # ========================================================
    # Remove genes completely
    # ========================================================

    adata = adata[:, []].copy()

    adata.layers = {}
    
    adata.raw = None

    # ========================================================
    # Reduce memory usage
    # ========================================================

    adata.obs["batch"] = (
        adata.obs["batch"]
        .astype("category")
    )

    adata.obs["celltype"] = (
        adata.obs["celltype"]
        .astype("category")
    )

    all_adatas.append(adata)

# ============================================================
# CONCATENATE EVERYTHING
# ============================================================

adata_global = ad.concat(
    all_adatas,
    join="outer",
    merge="same",
    fill_value=np.nan,
)

print("\n" + "=" * 70)
print("FINAL OBJECT")
print("=" * 70)

print(adata_global)

print("\nEmbeddings available:")

for k in adata_global.obsm.keys():

    print(
        k,
        adata_global.obsm[k].shape,
    )

In [ ]:
from tqdm.notebook import tqdm

CELLTYPE_MAP = {
    "Mesenchymal": "Mesenchymal",
    "Epithelial": "Epithelial",
    "Myeloid": "Myeloid",
    "Lymphoid": "Lymphoid",
    "Endothelial": "Endothelial",
    "Malignant": "Malignant",

    # Ambiguous / low-confidence labels
    "Myeloid/Lymphoid": "Other",
    "Myeloid/Mesenchymal": "Other",
    "Noise": "Other",
    "nan": "Other",
    np.nan: "Other",
}

adata_global.obs["celltype_unified"] = (
    adata_global.obs["celltype"]
    .astype(str)
    .map(CELLTYPE_MAP)
    .fillna("Other")
)

In [ ]:
# ============================================================
# Evaluate embeddings
# ============================================================

embedding_results = []

for emb_name in EMBEDDING_FILES.keys():

    embed_key = f"X_{emb_name}"

    if embed_key not in adata_global.obsm:
        continue

    print("\n" + "=" * 70)
    print(f"Evaluating {emb_name}")
    print("=" * 70)

    X = adata_global.obsm[embed_key]

    # --------------------------------------------------------
    # Compute neighbors
    # --------------------------------------------------------

    sc.pp.neighbors(
        adata_global,
        use_rep=embed_key,
        n_neighbors=15,
    )

    # --------------------------------------------------------
    # ARI / NMI
    # --------------------------------------------------------

    labels = adata_global.obs["celltype_unified"].values

    ari, nmi = compute_ari_nmi(
        X,
        labels,
    )

    # --------------------------------------------------------
    # Silhouette metrics
    # --------------------------------------------------------

    sil_metrics = compute_silhouette_metrics(
        adata_global,
        embed_key=embed_key,
        batch_key="batch",
        label_key="celltype_unified",
    )

    # --------------------------------------------------------
    # Store results
    # --------------------------------------------------------

    result = {
        "embedding": emb_name,
        "n_cells": adata_global.n_obs,
        "ARI": ari,
        "NMI": nmi,
        **sil_metrics,
    }

    embedding_results.append(result)

    print(result)

# ============================================================
# Convert to DataFrame
# ============================================================

results_df = pd.DataFrame(embedding_results)

# Optional sorting
results_df = results_df.sort_values(
    "ARI",
    ascending=False,
)

# Reset index
results_df = results_df.reset_index(drop=True)

# Display
display(results_df)

# ============================================================
# Save
# ============================================================

results_df.to_csv(
    "embedding_benchmark_results.csv",
    index=False,
)

print("\nSaved results to:")
print("embedding_benchmark_results.csv")

In [ ]:
# ============================================================
# UMAP Visualization using MiniBatchKMeans clusters
#
# Shows:
#   - celltype
#   - batch
#   - kmeans clusters
#
# for:
#   - scGPT
#   - STACK
#   - nicheformer
# ============================================================

import scanpy as sc
import matplotlib.pyplot as plt
from sklearn.cluster import MiniBatchKMeans

sc.set_figure_params(
    figsize=(6, 6),
    dpi=120,
)

for emb_name in EMBEDDING_FILES.keys():

    embed_key = f"X_{emb_name}"

    if embed_key not in adata_global.obsm:
        continue

    print("\n" + "=" * 70)
    print(f"Computing UMAP for {emb_name}")
    print("=" * 70)

    X = adata_global.obsm[embed_key]

    # --------------------------------------------------------
    # Compute neighbors
    # --------------------------------------------------------

    sc.pp.neighbors(
        adata_global,
        use_rep=embed_key,
        n_neighbors=15,
        key_added=f"neighbors_{emb_name}",
    )

    # --------------------------------------------------------
    # Compute UMAP
    # --------------------------------------------------------

    sc.tl.umap(
        adata_global,
        neighbors_key=f"neighbors_{emb_name}",
    )

    # Save coordinates
    adata_global.obsm[f"X_umap_{emb_name}"] = (
        adata_global.obsm["X_umap"].copy()
    )

    # ========================================================
    # MiniBatchKMeans clustering
    # SAME clustering used for ARI/NMI
    # ========================================================

    labels = adata_global.obs["celltype"].values

    n_clusters = len(np.unique(labels))

    km = MiniBatchKMeans(
        n_clusters=n_clusters,
        random_state=RANDOM_STATE,
        batch_size=4096,
        n_init="auto",
    )

    pred = km.fit_predict(X)

    cluster_key = f"kmeans_{emb_name}"

    adata_global.obs[cluster_key] = (
        pred.astype(str)
    )

    # ========================================================
    # Plot Cell Type
    # ========================================================

    sc.pl.embedding(
        adata_global,
        basis=f"X_umap_{emb_name}",
        color="celltype",
        title=f"{emb_name} — Cell Type",
        frameon=False,
        legend_loc="right margin",
        size=10,
    )

    # ========================================================
    # Plot Batch
    # ========================================================

    sc.pl.embedding(
        adata_global,
        basis=f"X_umap_{emb_name}",
        color="batch",
        title=f"{emb_name} — Batch",
        frameon=False,
        size=10,
    )

    # ========================================================
    # Plot KMeans Clusters
    # ========================================================

    sc.pl.embedding(
        adata_global,
        basis=f"X_umap_{emb_name}",
        color=cluster_key,
        title=f"{emb_name} — MiniBatchKMeans",
        frameon=False,
        legend_loc="right margin",
        size=10,
    )

# Use linear probe 

In [ ]:
import numpy as np
import torch
import torch.nn as nn

from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
)
from sklearn.utils.class_weight import compute_class_weight
from sklearn.preprocessing import normalize


def linear_probe_pytorch(
    X,
    labels,
    test_size=0.2,
    epochs=50,
    batch_size=4096,
    lr=1e-3,
    weight_decay=1e-4,
    random_state=42,
    use_class_weights=True,
    device=None,
):

    if device is None:
        device = (
            "cuda"
            if torch.cuda.is_available()
            else "cpu"
        )

    # --------------------------------------------------
    # Normalize embeddings
    # --------------------------------------------------

    X = normalize(
        X,
        norm="l2",
    )

    # --------------------------------------------------
    # Encode labels
    # --------------------------------------------------

    le = LabelEncoder()

    y = le.fit_transform(labels)

    n_classes = len(le.classes_)

    # --------------------------------------------------
    # Stratified split
    # --------------------------------------------------

    splitter = StratifiedShuffleSplit(
        n_splits=1,
        test_size=test_size,
        random_state=random_state,
    )

    train_idx, test_idx = next(
        splitter.split(X, y)
    )

    X_train = torch.tensor(
        X[train_idx],
        dtype=torch.float32,
    )

    X_test = torch.tensor(
        X[test_idx],
        dtype=torch.float32,
    )

    y_train = torch.tensor(
        y[train_idx],
        dtype=torch.long,
    )

    y_test = torch.tensor(
        y[test_idx],
        dtype=torch.long,
    )

    # --------------------------------------------------
    # Linear probe
    # --------------------------------------------------

    model = nn.Linear(
        X.shape[1],
        n_classes,
    ).to(device)

    # --------------------------------------------------
    # Class weights
    # --------------------------------------------------

    if use_class_weights:

        weights = compute_class_weight(
            class_weight="balanced",
            classes=np.unique(y),
            y=y,
        )

        weights = torch.tensor(
            weights,
            dtype=torch.float32,
            device=device,
        )

        criterion = nn.CrossEntropyLoss(
            weight=weights
        )

    else:

        criterion = nn.CrossEntropyLoss()

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay,
    )

    # --------------------------------------------------
    # Mini-batch training
    # --------------------------------------------------

    n_train = X_train.shape[0]

    for epoch in range(epochs):

        perm = torch.randperm(n_train)

        for i in range(
            0,
            n_train,
            batch_size,
        ):

            idx = perm[
                i:i + batch_size
            ]

            xb = (
                X_train[idx]
                .to(device)
            )

            yb = (
                y_train[idx]
                .to(device)
            )

            logits = model(xb)

            loss = criterion(
                logits,
                yb,
            )

            optimizer.zero_grad()

            loss.backward()

            optimizer.step()

    # --------------------------------------------------
    # Evaluate
    # --------------------------------------------------

    model.eval()

    with torch.no_grad():

        logits = model(
            X_test.to(device)
        )

        pred = (
            logits.argmax(1)
            .cpu()
            .numpy()
        )

    y_true = y_test.numpy()

    return {

        "accuracy":
            accuracy_score(
                y_true,
                pred,
            ),

        "macro_f1":
            f1_score(
                y_true,
                pred,
                average="macro",
            ),

        "balanced_accuracy":
            balanced_accuracy_score(
                y_true,
                pred,
            ),

        "n_classes":
            n_classes,
    }

In [ ]:
def integration_score(
    metric,
    n_classes,
):

    baseline = 1 / n_classes

    score = 1 - (
        (metric - baseline)
        /
        (1 - baseline)
    )

    return np.clip(
        score,
        0,
        1,
    )

In [ ]:
# ============================================================
# Evaluate embeddings with PyTorch Linear Probes
# ============================================================

embedding_results = []

for emb_name in EMBEDDING_FILES.keys():

    embed_key = f"X_{emb_name}"

    if embed_key not in adata_global.obsm:
        continue

    print("\n" + "=" * 80)
    print(f"Evaluating {emb_name}")
    print("=" * 80)

    # --------------------------------------------------------
    # Extract embedding
    # --------------------------------------------------------

    X = adata_global.obsm[embed_key]

    # --------------------------------------------------------
    # Keep only valid embeddings
    # --------------------------------------------------------

    valid_mask = ~np.isnan(X).all(axis=1)

    n_valid = valid_mask.sum()

    print(
        f"Using {n_valid:,} / "
        f"{adata_global.n_obs:,} cells"
    )

    if n_valid == 0:

        print("No valid embeddings")
        continue

    adata_valid = adata_global[valid_mask].copy()

    X_valid = X[valid_mask]

    print(
        f"Embedding shape: {X_valid.shape}"
    )

    # ========================================================
    # CELL TYPE PROBE
    # ========================================================

    print("\nRunning cell type probe...")

    celltype_results = linear_probe_pytorch(
        X_valid,
        adata_valid.obs[
            "celltype_unified"
        ].values,
        epochs=30,
        batch_size=8192,
        lr=1e-3,
        use_class_weights=True,
    )

    # ========================================================
    # BATCH PROBE
    # ========================================================

    print("\nRunning batch probe...")

    batch_results = linear_probe_pytorch(
        X_valid,
        adata_valid.obs[
            "batch"
        ].values,
        epochs=30,
        batch_size=8192,
        lr=1e-3,
        use_class_weights=True,
    )

    # ========================================================
    # Integration scores
    # ========================================================

    n_batches = batch_results["n_classes"]

    batch_integration_accuracy = (
        integration_score(
            batch_results["accuracy"],
            n_batches,
        )
    )

    batch_integration_f1 = (
        integration_score(
            batch_results["macro_f1"],
            n_batches,
        )
    )

    batch_integration_balanced_accuracy = (
        integration_score(
            batch_results[
                "balanced_accuracy"
            ],
            n_batches,
        )
    )

    # ========================================================
    # Store results
    # ========================================================

    result = {

        "embedding": emb_name,

        "n_cells":
            adata_valid.n_obs,

        # ----------------------------------
        # Cell type probe
        # ----------------------------------

        "celltype_accuracy":
            celltype_results[
                "accuracy"
            ],

        "celltype_macro_f1":
            celltype_results[
                "macro_f1"
            ],

        "celltype_balanced_accuracy":
            celltype_results[
                "balanced_accuracy"
            ],

        # ----------------------------------
        # Batch probe
        # ----------------------------------

        "batch_accuracy":
            batch_results[
                "accuracy"
            ],

        "batch_macro_f1":
            batch_results[
                "macro_f1"
            ],

        "batch_balanced_accuracy":
            batch_results[
                "balanced_accuracy"
            ],

        # ----------------------------------
        # Integration score
        # ----------------------------------

        "batch_integration_accuracy":
            batch_integration_accuracy,

        "batch_integration_f1":
            batch_integration_f1,

        "batch_integration_balanced_accuracy":
            batch_integration_balanced_accuracy,
    }

    embedding_results.append(result)

    print("\nResults:")
    print(result)

# ============================================================
# Convert to DataFrame
# ============================================================

results_df = pd.DataFrame(
    embedding_results
)

# ------------------------------------------------------------
# Optional overall score
# ------------------------------------------------------------

results_df["overall_score"] = (
    results_df["celltype_macro_f1"]
    *
    results_df["batch_integration_f1"]
)

# ------------------------------------------------------------
# Sort
# ------------------------------------------------------------

results_df = results_df.sort_values(
    [
        "overall_score",
    ],
    ascending=False,
)

results_df = (
    results_df
    .reset_index(drop=True)
)

# ============================================================
# Display
# ============================================================

display(results_df)

# ============================================================
# Save
# ============================================================

results_df.to_csv(
    "embedding_linear_probe_results.csv",
    index=False,
)

print(
    "\nSaved results to:\n"
    "embedding_linear_probe_results.csv"
)

In [ ]:
results_df = pd.read_csv('embedding_linear_probe_results.csv')

# Plot

In [ ]:
import matplotlib.pyplot as plt

# --------------------------------------------------
# Figure
# --------------------------------------------------

fig, ax = plt.subplots(
    figsize=(5, 5),
    dpi=300,
)

# --------------------------------------------------
# Colors / markers
# --------------------------------------------------

style_map = {
    "scGPT": {
        "color": "#1f77b4",
        "marker": "o",
    },
    "STACK": {
        "color": "#d62728",
        "marker": "s",
    },
    "nicheformer": {
        "color": "#2ca02c",
        "marker": "^",
    },
}

# --------------------------------------------------
# Plot points
# --------------------------------------------------

for _, row in results_df.iterrows():

    emb = row["embedding"]

    ax.scatter(
        row["batch_integration_f1"],
        row["celltype_macro_f1"],
        s=180,
        marker=style_map[emb]["marker"],
        color=style_map[emb]["color"],
        edgecolor="black",
        linewidth=1.0,
        zorder=3,
        label=emb,
    )

    ax.annotate(
        emb,
        (
            row["batch_integration_f1"],
            row["celltype_macro_f1"],
        ),
        xytext=(8, 4),
        textcoords="offset points",
        fontsize=11,
        fontweight="bold",
    )

# --------------------------------------------------
# Labels
# --------------------------------------------------

ax.set_xlabel(
    "Batch Integration Score (BIS)",
    fontsize=14,
)

ax.set_ylabel(
    "cell-type (ct)F1",
    fontsize=14,
)

ax.set_title(
    "Embedding Benchmark",
    fontsize=14,
    fontweight="bold",
)

# --------------------------------------------------
# Grid
# --------------------------------------------------

ax.grid(
    alpha=0.25,
    linestyle="--",
)

# --------------------------------------------------
# Remove top/right spines
# --------------------------------------------------

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

# Thicken remaining spines
ax.spines["left"].set_linewidth(1.2)
ax.spines["bottom"].set_linewidth(1.2)

# --------------------------------------------------
# Limits with padding
# --------------------------------------------------

x = results_df["batch_integration_f1"]
y = results_df["celltype_macro_f1"]

ax.set_xlim(
    x.min() - 0.05,
    x.max() + 0.05,
)

ax.set_ylim(
    y.min() - 0.05,
    y.max() + 0.05,
)

# --------------------------------------------------
# Legend
# --------------------------------------------------

handles, labels = ax.get_legend_handles_labels()

by_label = dict(zip(labels, handles))

ax.legend(
    by_label.values(),
    by_label.keys(),
    frameon=False,
    loc="lower left",
)

plt.tight_layout()

plt.show()